# Verificação Facial e Avaliação de Robustez contra Deepfakes

Este notebook documenta o pipeline completo de **verificação facial** e **avaliação de vulnerabilidade a ataques de face swap** utilizando:

- **YOLOv8-Face** — detecção de faces + extração de 5 landmarks
- **AdaFace (IR-50)** — extração de embeddings faciais (512-d)
- **FaceFusion (inswapper_128)** — geração de deepfakes por face swap

## Índice

1. Configuração do Ambiente Docker
2. Teste da GPU
3. Dataset LFW
4. Arquitetura: YOLOv8 + AdaFace
5. Baseline LFW (sem manipulação)
6. Resultados do Baseline
7. Ataque Deepfake — Dataset Pessoal
8. **Ataque Deepfake — LFW (30 swaps)**
9. **Análise Comparativa: Baseline vs Ataque**
10. **Organização dos Resultados para o Paper**

## Estrutura do Projeto

```
deepfaketeste/
├── Dockerfile                        # CUDA 12.2 + Python 3.10
├── docker-compose.gpu.yml            # Compose com GPU
├── requirements.txt
├── pipeline_lfw_adaface.ipynb        # Este notebook
├── facefusion/                       # Motor de face swap
├── app/
│   ├── baseline_lfw.py               # Baseline LFW
│   ├── net.py                        # Backbone AdaFace IR-50
│   ├── api.py                        # API FastAPI
│   ├── run_experiment.py             # Comparação cruzada dataset pessoal
│   ├── run_swaps_experiment.py       # Ataque deepfake (dataset pessoal)
│   ├── run_swaps_lfw.py             # Ataque deepfake (LFW)
│   └── results/                      # Resultados gerados
└── data/
    ├── lfw/                          # Dataset + swaps LFW
    ├── models/                       # Pesos dos modelos
    └── dataset_pessoal/              # Fotos pessoais + swaps
```

## Requisitos
- Linux com Docker instalado
- GPU NVIDIA com suporte CUDA
- NVIDIA Container Toolkit configurado

---
## Parte 1 — Configuração do Ambiente Docker

Todo o projeto roda dentro de um container Docker com suporte a GPU NVIDIA.

### 1.1 Dockerfile

O Dockerfile usa a imagem oficial da NVIDIA com CUDA 12.2.2 + cuDNN 8 sobre Ubuntu 22.04.

> **Pontos importantes:**
> - `libgl1` e `libglib2.0-0` são necessários para o OpenCV funcionar dentro do container
> - `ffmpeg` e `curl` são necessários para o FaceFusion
> - `CMD ["bash"]` mantém o container rodando em modo interativo

```dockerfile
FROM nvidia/cuda:12.2.2-cudnn8-runtime-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive

WORKDIR /app

RUN apt update && apt install -y \
    python3.10 \
    python3-pip \
    python3.10-dev \
    build-essential \
    git \
    libgl1 \
    libglib2.0-0 \
    curl \
    ffmpeg \
    && rm -rf /var/lib/apt/lists/*

RUN ln -s /usr/bin/python3.10 /usr/bin/python

COPY requirements.txt .
RUN pip install --upgrade pip
RUN pip install -r requirements.txt

COPY . .
CMD ["bash"]
```

### 1.2 docker-compose.gpu.yml

```yaml
services:
  ia:
    image: carolinavilela/deepfaketeste:v1.0
    container_name: ia_gpu
    runtime: nvidia
    network_mode: host
    volumes:
      - .:/app
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: all
              capabilities: [ gpu ]
    environment:
      - NVIDIA_VISIBLE_DEVICES=all
      - NVIDIA_DRIVER_CAPABILITIES=all
    tty: true
    stdin_open: true
```

### 1.3 Comandos para subir o container

```bash
# 1. Subir o container em segundo plano
sudo docker-compose -f docker-compose.gpu.yml up -d

# 2. Verificar se o container está rodando
sudo docker ps

# 3. Entrar no container
sudo docker exec -it ia_gpu bash
```

> **Dica:** Se aparecer erro de container com nome duplicado:
> ```bash
> sudo docker rm -f ia_gpu
> ```

---
## Parte 2 — Teste da GPU

Antes de rodar qualquer experimento, verifique se a GPU está acessível dentro do container.

In [ ]:
# app/test_gpu.py
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA disponível:", torch.cuda.is_available())
print("Device usado:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

x = torch.randn(3, 3).to(device)
print("Tensor está em:", x.device)
print(x)

Saída esperada:
```
CUDA disponível: True
Device usado: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Tensor está em: cuda:0
```

---
## Parte 3 — Dataset LFW

O **LFW (Labeled Faces in the Wild)** é o benchmark padrão para verificação facial.

- **13.233 imagens** de rostos de celebridades coletadas da web
- **5.749 identidades** diferentes
- Protocolo padrão: **6.000 pares** (3.000 positivos + 3.000 negativos) divididos em 10 folds

---
## Parte 4 — Arquitetura: YOLOv8 + AdaFace

### Pipeline de Verificação Facial

```
Imagem A ─┐
           ├─► YOLOv8-face ─► Landmarks ─► Alinhamento ─► AdaFace ─► Embedding A ─┐
Imagem B ─┘                                                                         ├─► Similaridade Cosseno ─► Decisão
                                     Landmarks ─► Alinhamento ─► AdaFace ─► Embedding B ─┘
```

| Componente | Descrição |
|---|---|
| **YOLOv8n-face** | Detecta rostos + extrai 5 landmarks (olhos, nariz, boca) |
| **Alinhamento Facial** | Transformação afim usando os landmarks para alinhar o rosto |
| **AdaFace IR-50** | Rede residual profunda que extrai embedding 512-d do rosto |
| **Cosine Similarity** | Mede similaridade entre dois embeddings (-1 a 1) |
| **Threshold** | Define o limiar de decisão: acima = mesma pessoa |

### Por que AdaFace?
O AdaFace adapta a função de perda de acordo com a **qualidade da imagem**: imagens de baixa qualidade recebem menor peso no treinamento. Isso o torna robusto para cenários do mundo real.

### 4.1 Arquitetura do AdaFace — net.py

O arquivo `app/net.py` implementa o **Backbone IResNet** do AdaFace, compatível com os checkpoints oficiais.

> **Atenção:** A ordem das camadas no bloco residual deve ser exatamente:
> `BN → Conv → BN → PReLU → Conv → BN`

Estrutura geral do modelo:
- `input_layer`: Conv3×3 → BN → PReLU
- `body`: 24 blocos residuais (IR-50 = [3, 4, 14, 3] blocos por estágio)
- `output_layer`: BN → Dropout → Flatten → Linear(512×7×7 → 512) → BN

### 4.2 Detecção de Face com Alinhamento (5 Landmarks)

A detecção utiliza os **5 pontos faciais** extraídos pelo YOLOv8-face para aplicar uma **transformação afim** que alinha o rosto segundo pontos de referência padrão do AdaFace:

```python
REFERENCE_FACIAL_POINTS = np.array([
    [38.2946, 51.6963],  # Olho esquerdo
    [73.5318, 51.5014],  # Olho direito
    [56.0252, 71.7366],  # Nariz
    [41.5493, 92.3655],  # Canto esquerdo da boca
    [70.7299, 92.2041]   # Canto direito da boca
], dtype=np.float32)
```

O alinhamento usa `cv2.estimateAffinePartial2D` + `cv2.warpAffine` para gerar uma imagem 112×112 perfeitamente normalizada.

---
## Parte 5 — Baseline LFW (sem manipulação)

O script `app/baseline_lfw.py` implementa o pipeline completo de avaliação no protocolo padrão LFW.

### Execução
```bash
# Dentro do container:
python app/baseline_lfw.py
```

In [ ]:
# Código simplificado do baseline (o completo está em app/baseline_lfw.py)
import os, sys, time, numpy as np, cv2, torch
import torch.nn.functional as F
from sklearn.metrics import roc_curve, auc, accuracy_score

sys.path.insert(0, 'app')
from baseline_lfw import (
    ADAFACE_CKPT, YOLO_FACE_PATH, FACE_SIZE, DEVICE, LFW_DIR, PAIRS_FILE,
    load_adaface, detect_and_crop_face, preprocess_face, get_embedding,
    parse_pairs, ensure_lfw_images, find_best_threshold
)
from ultralytics import YOLO

# Carregar modelos
yolo = YOLO(YOLO_FACE_PATH, verbose=False)
adaface = load_adaface(ADAFACE_CKPT).to(DEVICE)
print(f"Modelos carregados! Device: {DEVICE}")

---
## Parte 6 — Resultados do Baseline LFW

### Métricas Principais

| Métrica | Valor |
|---------|-------|
| Pares avaliados | 6.000 (3.000 genuínos + 3.000 impostores) |
| Acurácia | **95.97%** (threshold ótimo = 0.238) |
| AUC | **0.9660** |
| EER | **6.47%** |

### FAR / FRR nos Thresholds do Artigo

| Threshold (τ) | FAR (%) | FRR (%) |
|:--------------:|:-------:|:-------:|
| 0.45 | 0.00 | 12.40 |
| 0.60 | 0.00 | 38.13 |

### Acurácia por Fold (10-fold cross-validation)

| Fold | Acurácia |
|:----:|:--------:|
| 1 | 96.50% |
| 2 | 95.33% |
| 3 | 93.67% |
| 4 | 96.83% |
| 5 | 96.33% |
| 6 | 96.33% |
| 7 | 96.17% |
| 8 | 97.67% |
| 9 | 94.50% |
| 10 | 96.33% |
| **Média** | **95.97% ± 1.11%** |

### Gráficos Gerados

Os seguintes gráficos são gerados automaticamente em `app/results/`:

- `score_hist_baseline.png` — Histograma: distribuição de scores genuínos vs. impostores
- `roc_curve_lfw.png` — Curva ROC com AUC e ponto de EER

### Configuração Experimental

| Parâmetro | Valor |
|-----------|-------|
| Hardware | NVIDIA GeForce RTX 4050 Laptop GPU |
| OS | Linux 6.17.0-20-generic |
| Python | 3.10.12 |
| PyTorch | 2.11.0+cu130 |
| Ultralytics | 8.4.35 |
| Tempo total | 127.3s |

---
## Parte 7 — Ataque Deepfake: Dataset Pessoal

O script `app/run_swaps_experiment.py` realiza o ataque completo no dataset pessoal:

1. Seleciona pares de pessoas do diretório `data/dataset_pessoal/originais/`
2. Usa o FaceFusion (inswapper_128) para colar o rosto da Pessoa A na foto da Pessoa B
3. Compara o swap com a foto original da Pessoa A usando AdaFace
4. Se similaridade ≥ threshold → deepfake enganou a IA

### Execução
```bash
# Dentro do container:
python app/run_swaps_experiment.py
```

### Pares Testados

| Rosto (Source) | Corpo (Target) | Resultado |
|----------------|----------------|:---------:|
| Paulo (sério, escuro) | João (sério) | Swap gerado |
| Lucas (sério, claro) | Carol (séria, claro) | Swap gerado |
| Carol (séria, escuro) | Paulo (sorrindo, claro) | Swap gerado |

Os swaps ficam salvos em `data/dataset_pessoal/swaps/`.

In [ ]:
# Executando o experimento de swap no dataset pessoal:
# Use este comando no terminal docker (sudo docker exec -it ia_gpu bash):

!python app/run_swaps_experiment.py

---
## Parte 8 — Ataque Deepfake: LFW (30 swaps)

O script `app/run_swaps_lfw.py` aplica o **mesmo raciocínio** do dataset pessoal, mas agora no benchmark acadêmico LFW:

### Metodologia

1. **Seleção de identidades**: filtra 610 identidades do LFW que possuem ≥ 4 fotos
2. **Geração de pares cruzados**: seleciona aleatoriamente 30 pares de pessoas diferentes (seed=42 para reprodutibilidade)
3. **Face swap**: usa FaceFusion (inswapper_128) para colar o rosto da Pessoa A na foto da Pessoa B
4. **Avaliação biométrica**: compara o swap com a foto original da Pessoa A usando AdaFace
5. **Veredicto**: se similaridade ≥ threshold → deepfake enganou o sistema

### Execução
```bash
# Dentro do container:
python app/run_swaps_lfw.py
```

In [ ]:
# Executando o experimento de swap no LFW:
!python app/run_swaps_lfw.py

### Resultados do Ataque — LFW

#### Taxa de Sucesso do Ataque (Attack Success Rate — ASR)

| Threshold (τ) | Swaps que Enganaram | ASR |
|:--------------:|:-------------------:|:---:|
| **0.45** | 28/30 | **93.3%** |
| **0.60** | 27/30 | **90.0%** |

#### Estatísticas de Similaridade dos Swaps

| Estatística | Valor |
|-------------|:-----:|
| Média | 0.6846 |
| Mediana | 0.7565 |
| Desvio Padrão | — |
| Mínimo | -0.1167 |
| Máximo | 0.8232 |

#### Detalhes por Swap

| # | Pessoa Alvo (Rosto) | Corpo Usado | Similaridade | τ=0.45 | τ=0.60 |
|:-:|---------------------|-------------|:------------:|:------:|:------:|
| 1 | Clint Eastwood | Amelie Mauresmo | 0.5277 | 🚨 | 🛡️ |
| 2 | Jay Garner | Hu Jintao | 0.6762 | 🚨 | 🚨 |
| 3 | Christine Baumgartner | Tom Hanks | 0.7455 | 🚨 | 🚨 |
| 4 | Padraig Harrington | Angela Bassett | 0.8000 | 🚨 | 🚨 |
| 5 | Hipolito Mejia | Jacques Chirac | 0.7981 | 🚨 | 🚨 |
| 6 | Gregg Popovich | Tom Hanks | 0.6722 | 🚨 | 🚨 |
| 7 | Queen Rania | Yoko Ono | 0.8232 | 🚨 | 🚨 |
| 8 | Ernie Els | Padraig Harrington | 0.7644 | 🚨 | 🚨 |
| 9 | Emma Watson | Helen Clark | 0.7525 | 🚨 | 🚨 |
| 10 | Celine Dion | Mick Jagger | 0.7646 | 🚨 | 🚨 |
| 11 | Madonna | Jim Hahn | 0.7469 | 🚨 | 🚨 |
| 12 | Tim Allen | David Heymann | 0.7652 | 🚨 | 🚨 |
| 13 | Tony Blair | Joseph Biden | 0.7778 | 🚨 | 🚨 |
| 14 | Gloria Trevi | Bob Hope | 0.7241 | 🚨 | 🚨 |
| 15 | Joschka Fischer | Calista Flockhart | 0.6674 | 🚨 | 🚨 |
| 16 | Felipe Perez Roque | Michael Chang | -0.1167 | 🛡️ | 🛡️ |
| 17 | Jiri Novak | Bob Stoops | 0.7677 | 🚨 | 🚨 |
| 18 | Fujio Cho | Thomas Fargo | 0.6126 | 🚨 | 🚨 |
| 19 | Fernando Gonzalez | Richard Armitage | 0.7073 | 🚨 | 🚨 |
| 20 | Vaclav Havel | Hitomi Soga | -0.0257 | 🛡️ | 🛡️ |
| 21 | Igor Ivanov | Angela Bassett | 0.7829 | 🚨 | 🚨 |
| 22 | Joan Laporta | Bill Simon | 0.7913 | 🚨 | 🚨 |
| 23 | Vincent Brooks | Kim Ryong-sung | 0.7129 | 🚨 | 🚨 |
| 24 | Sarah Hughes | Muhammad Ali | 0.7093 | 🚨 | 🚨 |
| 25 | Doris Schroeder | Jim Tressel | 0.7844 | 🚨 | 🚨 |
| 26 | Vanessa Redgrave | Tim Henman | 0.7697 | 🚨 | 🚨 |
| 27 | Paul Burrell | Xavier Malisse | 0.7606 | 🚨 | 🚨 |
| 28 | Hitomi Soga | Dick Cheney | 0.7410 | 🚨 | 🚨 |
| 29 | Catherine Zeta-Jones | Ashanti | 0.7687 | 🚨 | 🚨 |
| 30 | Mick Jagger | John Kerry | 0.7674 | 🚨 | 🚨 |

> 🚨 = Deepfake Enganou a IA | 🛡️ = Deepfake Barrado

---
## Parte 9 — Análise Comparativa: Baseline vs Ataque

### Como interpretar os resultados

O gráfico `lfw_baseline_vs_attack.png` compara a distribuição dos scores em três cenários:

| Distribuição | Cor | Significado |
|---|---|---|
| **Genuínos** | 🔵 Azul | Pares verdadeiros (mesma pessoa) — scores altos |
| **Impostores** | 🟢 Verde | Pares falsos (pessoas diferentes) — scores baixos |
| **Deepfakes/Swaps** | 🔴 Vermelho | Ataques de face swap — **deveriam ter scores baixos, mas...** |

### Tabela Comparativa — Baseline vs Ataque

| Cenário | Similaridade Média | τ=0.45 Aceito? | τ=0.60 Aceito? |
|---------|:-----------------:|:--------------:|:--------------:|
| **Genuínos (baseline)** | ~0.45-0.60 | ✅ Maioria sim | ⚠ Parte recusada |
| **Impostores (baseline)** | ~0.00-0.15 | ✅ 0% aceito (FAR=0%) | ✅ 0% aceito |
| **Deepfakes (ataque)** | **0.6846** | ❌ **93.3% aceitos** | ❌ **90.0% aceitos** |

### Conclusão Crítica

> **Os deepfakes apresentam scores de similaridade na mesma faixa dos pares genuínos (0.60-0.82).** 
> Isso significa que o sistema biométrico baseado apenas em AdaFace IR-50 é **altamente vulnerável** a ataques de face swap.
> 
> O FAR de **0%** no baseline (sem manipulação) sobe para **93.3%** quando os impostores usam deepfakes como ferramenta de ataque.
> 
> Isso evidencia a necessidade de **mecanismos complementares** como **Liveness Detection** (Detecção de Vivacidade) para proteger sistemas biométricos contra fraudes sofisticadas.

---
## Parte 10 — Organização dos Resultados para o Paper

### Arquivos Gerados

| Arquivo | Descrição | Usar no Paper? |
|---------|-----------|:--------------:|
| `app/results/lfw_scores.csv` | Scores de todos os 6.000 pares do baseline | Tabela de métricas |
| `app/results/lfw_swap_attack_results.csv` | Detalhes dos 30 swaps no LFW | Tabela de ataque |
| `app/results/score_hist_baseline.png` | Distribuição genuínos vs impostores | **Figura no paper** |
| `app/results/roc_curve_lfw.png` | Curva ROC (AUC + EER) | **Figura no paper** |
| `app/results/lfw_swap_attack_histogram.png` | Scores dos deepfakes | **Figura no paper** |
| `app/results/lfw_baseline_vs_attack.png` | Baseline vs Ataque combinados | **Figura principal** |
| `app/results/experimental_config.txt` | Hardware, versões, config | Seção de Metodologia |
| `app/results/baseline_conclusion.txt` | Conclusão do baseline | Referência |
| `app/results/lfw_swap_attack_conclusion.txt` | Conclusão do ataque | Referência |

### Estrutura Sugerida do Paper

```
1. Introdução
   - Contexto: biometria facial e ameaças de deepfakes
   - Objetivo: avaliar robustez do AdaFace contra face swaps

2. Trabalhos Relacionados
   - AdaFace, ArcFace, FaceNet
   - Ataques de deepfake (face swap, reenactment)
   - Defesas (liveness detection, anti-spoofing)

3. Metodologia
   3.1 Pipeline de Verificação Facial
       → Diagrama do pipeline (YOLOv8 + AdaFace)
       → Tabela com configuração experimental
   3.2 Geração de Deepfakes
       → FaceFusion (inswapper_128)
       → Descrição dos datasets (pessoal + LFW)
   3.3 Protocolo de Avaliação
       → Métricas: FAR, FRR, ASR, AUC, EER
       → Thresholds: τ = 0.45 e τ = 0.60

4. Resultados
   4.1 Baseline LFW (sem manipulação)
       → Tabela de métricas
       → Curva ROC (roc_curve_lfw.png)
       → Histograma de scores (score_hist_baseline.png)
   4.2 Ataque Deepfake — Dataset Pessoal
       → Tabela com resultados dos swaps
   4.3 Ataque Deepfake — LFW
       → Tabela de ASR por threshold
       → Histograma de ataque (lfw_swap_attack_histogram.png)
   4.4 Análise Comparativa
       → Gráfico baseline vs ataque (lfw_baseline_vs_attack.png)
       → Discussão sobre vulnerabilidade

5. Discussão
   - Por que o AdaFace é vulnerável?
   - Limitações do estudo (N=30 swaps, 1 modelo de swap)
   - Necessidade de Liveness Detection

6. Conclusão
   - Sistema biométrico vulnerável: ASR = 93.3% (τ=0.45)
   - Deepfakes caem na mesma região de scores genuínos
   - Recomendação: combinar verificação facial + liveness

7. Referências
```

### Tabelas Prontas para o Paper

#### Tabela 1 — Configuração Experimental

| Parâmetro | Valor |
|-----------|-------|
| Detecção Facial | YOLOv8n-face |
| Embedding Facial | AdaFace IR-50 (WebFace4M) |
| Dimensão do Embedding | 512 |
| Tamanho da Face (input) | 112 × 112 |
| Motor de Face Swap | FaceFusion 3.x (inswapper_128) |
| Hardware | NVIDIA RTX 4050 Laptop (CUDA) |
| Dataset de Avaliação | LFW (6.000 pares) |
| Nº de Swaps Gerados (LFW) | 30 |
| Thresholds Analisados | τ = 0.45, τ = 0.60 |

#### Tabela 2 — Resultados do Baseline (sem ataque)

| Métrica | Valor |
|---------|-------|
| Acurácia | 95.97% |
| AUC | 0.9660 |
| EER | 6.47% |
| FAR (τ=0.45) | 0.00% |
| FRR (τ=0.45) | 12.40% |
| FAR (τ=0.60) | 0.00% |
| FRR (τ=0.60) | 38.13% |

#### Tabela 3 — Resultados do Ataque Deepfake (LFW)

| Threshold | ASR | Swaps Aceitos | Swaps Barrados |
|:---------:|:---:|:-------------:|:--------------:|
| τ = 0.45 | **93.3%** | 28/30 | 2/30 |
| τ = 0.60 | **90.0%** | 27/30 | 3/30 |

#### Tabela 4 — Impacto do Ataque nos Indicadores

| Indicador | Baseline (sem ataque) | Com Ataque Deepfake | Variação |
|-----------|:---------------------:|:-------------------:|:--------:|
| FAR (τ=0.45) | 0.00% | **93.3%** (ASR) | +93.3 p.p. |
| FAR (τ=0.60) | 0.00% | **90.0%** (ASR) | +90.0 p.p. |
| Sim. Média (impostor) | ~0.05 | **0.6846** (swap) | +0.63 |

> **ASR = Attack Success Rate**: taxa de swaps que enganaram o sistema biométrico.

---
## Resumo Final do Pipeline Completo

```
1. Docker (NVIDIA CUDA 12.2.2 + cuDNN8 + Ubuntu 22.04)
   └─ Python 3.10 + PyTorch + Ultralytics + OpenCV

2. Datasets
   ├─ LFW: 13.233 imagens, 5.749 identidades, 6.000 pares
   └─ Pessoal: fotos controladas de membros do grupo

3. Detecção de Face
   └─ YOLOv8n-face → 5 landmarks → Alinhamento Afim → 112×112

4. Embedding Facial
   └─ AdaFace IR-50 (WebFace4M) → vetor 512-d normalizado

5. Verificação
   └─ Similaridade Cosseno → threshold → Mesma/Diferente pessoa

6. Geração de Deepfakes
   └─ FaceFusion (inswapper_128) → Face Swap automatizado

7. Avaliação de Robustez
   ├─ Baseline: Acurácia 95.97%, AUC 0.966, EER 6.47%
   ├─ Ataque τ=0.45: ASR = 93.3% (28/30 enganaram)
   └─ Ataque τ=0.60: ASR = 90.0% (27/30 enganaram)

8. Conclusão
   └─ Sistema VULNERÁVEL → necessita Liveness Detection
```